# Spotify Recommender System – Part 3: Data Preparation for Collaborative Models
**Author:** Miguel Vásquez  
**Date:** 23 September 2025
**Updated:** 04 September 2025

---

## 1. Introduction  

In earlier experiments, we attempted to train both **ALS** and **LightFM** in a single notebook.  
However, running multiple large-scale models sequentially proved unstable and often caused kernel crashes.  

Since **ALS** was selected as our primary model (see Notebook 1 for details), this notebook focuses on preparing the data pipeline that ensures stable training and reproducibility.  

To achieve this, we reorganized our workflow into modular steps:  

- **Data preparation (this notebook)**: generate and persist all intermediate artifacts required for ALS training.  
- **Model-specific notebooks**: use these artifacts for training and evaluation.  

This modular approach has two benefits:  
1. Avoids memory conflicts when working with large interaction matrices.  
2. Ensures that all experiments rely on the same preprocessed and reproducible data split.

---

## 2. Objectives of this notebook  

- Load the preprocessed Spotify playlist subset.  
- Build the **user–item interaction matrix**.  
- Perform a reproducible **train/test split** at the user level.  
- Persist all artifacts (interaction matrices and index mappings) for downstream models.  

In [1]:
import os
from IPython.display import display, Markdown

# ensure working dir (if you run from notebooks/)
if os.getcwd().endswith("notebooks"):
    os.chdir("..")
    display(Markdown(f"Changed working dir to {os.getcwd()}"))

Changed working dir to c:\Users\Miguel\portfolio\Spotify-Recommender-System

## 3. Data Preparation  

In this step, we transform the raw playlist–track dataset into a format suitable for collaborative filtering models.  
Rather than embedding all code here, we moved the logic into a reusable script: `src/save_data.py`.  

This script handles three key tasks:

### 3.1 Define unique user–item keys  
- Assign continuous indices (`user2idx`, `item2idx`) to map playlists and tracks.  
- Persist these mappings for consistent usage across models.  

### 3.2 Build the interaction matrix  
- Construct a **sparse matrix** `interactions` with shape `(n_users × n_items)`.  
- Each entry `(u, i)` stores the count of times user *u* added track *i*.  

### 3.3 Train–Test Split  

In earlier experiments, we applied a **random 80/20 split** of user–item interactions.  
However, this approach can introduce **information leakage**: a user may appear in both train and test sets with overlapping preferences, artificially inflating evaluation metrics.  

To address this, we adopt a **Leave-One-Out (LOO) strategy**:  

- For each user (playlist), exactly **one track** is set aside for testing.  
- All remaining tracks stay in the training set.  
- Users with only one interaction are fully retained in training (avoiding cold-start issues).  

---

#### Why LOO is better than random splits  

| User | Random 80/20 Split | Leave-One-Out Split |
|------|---------------------|----------------------|
| U1: [A, B, C, D] | Train: [A, B, C] <br> Test: [D] | Train: [A, B, C] <br> Test: [D] |
| U2: [X, Y]       | Train: [X] <br> Test: [Y]       | Train: [X] <br> Test: [Y] |
| U3: [Z]          | Train: [] <br> Test: [Z] ❌     | Train: [Z] <br> Test: [] ✅ |

**Advantages of LOO:**  
- Each user contributes with at least one interaction in training.  
- Test items always correspond to *real held-out interactions* rather than random subsets.  
- Evaluation becomes more interpretable: we measure how well the model can recover a missing track for a known user.  


---

## 4. Save Artifacts  

The following artifacts are generated and stored in `data/processed/`:  

- `interactions.npz` – full user–item matrix  
- `train_interactions.npz` – training matrix (LOO strategy)  
- `test_interactions.npz` – test matrix (LOO strategy)  
- `user2idx.pkl` / `item2idx.pkl` – index mappings  
- `tracks_processed.parquet` – cleaned track metadata  

In [2]:
from src.save_data import build_and_save_split_safe
if not os.path.exists("data/processed/train_interactions.npz"):
    build_and_save_split_safe()

## 5. Verification of Artifacts  

Before moving to model training, we validate the artifacts to ensure consistency:  

In [3]:
from scipy.sparse import load_npz
import pickle

# Load interaction matrices
train_csr = load_npz("data/processed/train_interactions.npz")
test_csr = load_npz("data/processed/test_interactions.npz")
interactions = load_npz("data/processed/interactions.npz")

# Load mappings
with open("data/processed/user2idx.pkl", "rb") as f:
    user2idx = pickle.load(f)
with open("data/processed/item2idx.pkl", "rb") as f:
    item2idx = pickle.load(f)

display(Markdown(f"Train shape: {train_csr.shape}"))
display(Markdown(f"Test shape: {test_csr.shape}"))
display(Markdown(f"Interactions shape: {interactions.shape}"))
display(Markdown(f"Users: {len(user2idx)}, Items: {len(item2idx)}"))

Train shape: (500000, 1614034)

Test shape: (500000, 1614034)

Interactions shape: (500000, 1614034)

Users: 500000, Items: 1614034

## 6. Conclusion

In this notebook, we:
- Defined reproducible **user and item indices**.
- Built the **interaction matrix** in CSR format.
- Performed a **Leave-One-Out train/test split** to avoid leakage.
- Persisted all artifacts needed for downstream experiments.

These artifacts now serve as the **fixed foundation** for collaborative models.
By centralizing preprocessing here, we guarantee that **ALS, LightFM, and future hybrid methods** all operate on the exact same data.

---

## 7. Next Steps  

With the data preparation finalized, the next phase of the project will focus on **training collaborative filtering models with ALS**, making use of the fixed artifacts created here.  

Future steps include:  
1. **ALS training and hyperparameter tuning** using Optuna.  
2. **Evaluation against baselines** (popularity models) to measure relative improvements.  
3. **Iterative refinement** of evaluation metrics and experimental design.  

This ensures a reproducible pipeline where all models share the same foundation and data split, avoiding leakage and inconsistencies.